# Assignment 04 — Weather AUS (Dataset: weatherAUS.csv)

EDA, preprocessing, and a model to predict RainTomorrow.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import joblib
import os

print('imports ok')

def ensure_models_dir():
    os.makedirs('models', exist_ok=True)


In [ ]:
# Load dataset

df = pd.read_csv('weatherAUS.csv')
print('shape:', df.shape)
df.head()

# Target
if 'RainTomorrow' in df.columns:
    target = 'RainTomorrow'
elif 'raintomorrow' in df.columns:
    target = 'raintomorrow'
else:
    raise ValueError('RainTomorrow column not found')

# Basic EDA
print('\nTarget distribution:')
print(df[target].value_counts(dropna=False))
print('\nMissing values per column:')
print(df.isna().sum().sort_values(ascending=False).head(20))

# Preprocessing: simple pipeline
X = df.drop(columns=[target])
y = df[target].map({'Yes':1,'No':0})

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

num_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[('num', num_transformer, num_cols), ('cat', cat_transformer, cat_cols)])

model = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('\nAccuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred, zero_division=0))

ensure_models_dir()
joblib.dump(model, 'models/weather_model.joblib')
print('\nSaved model to models/weather_model.joblib')
